# Travel Reimbursement Approval Agent
**Shaswat Prakash · Candidate assignment · USD**

> **Milestone 1 checkpoint — foundation only. Not ready for final submission.**
> This version validates the exact supplied policy, the five sample claims, and the required output contract. The agent, MCP integration, generated decisions, dashboard, and final results cell are added at subsequent milestones.

## README — setup and execution

Open this file in JupyterLab or Google Colab using **Python 3.11 or later** and choose **Run All**. This checkpoint is self-contained: it does not import project files, read an external policy file, or require an API key.

The first code cell checks the runtime and installs the pinned Pydantic dependency only if needed. Initial dependency installation requires internet access. A preconfigured environment avoids that installation step. No prompts or manual intervention are required during execution.

**Planned live execution:** LangChain with the local `qwen3.5:4b` model in Ollama. No OpenAI or LangSmith key is required. The completed notebook will default to clearly labeled replay of genuine captured model interactions so a reviewer does not need Ollama. `TRAVEL_MODE=live` will opt into a local live run; `TRAVEL_MODEL` will select the local model. These settings are not used by the foundation checkpoint.

**Submission boundary:** this notebook is the assessed artifact. Development scripts, the later companion application, and private study notes remain outside the submission folder. All employee names and claims below are the mock inputs supplied with the assignment.


In [1]:
import importlib.metadata
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11+ kernel before running this notebook.")
try:
    installed_pydantic = importlib.metadata.version("pydantic")
except importlib.metadata.PackageNotFoundError:
    installed_pydantic = None
if installed_pydantic != "2.13.5":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pydantic==2.13.5"])
print("Foundation runtime ready: Python", sys.version.split()[0], "· Pydantic", importlib.metadata.version("pydantic"))


Foundation runtime ready: Python 3.12.13 · Pydantic 2.13.5


## Appendix A — authoritative policy and limits

The following twelve `POL-*` identifiers and rule text are transcribed from **Appendix A, page 3** of the supplied *AI Developer Candidate Assignment — Travel Reimbursement Approval Agent*. Formatting is normalized; policy amounts, receipt rules, submission window, and approval tiers are preserved.

The approval thresholds apply to the **reimbursable amount after deductions**, not the gross claimed amount. A fully compliant claim in the manager tier is treated as approvable, as the assignment instructs.

## Appendix B — the five supplied claims

The embedded `SAMPLE_CLAIMS` preserve the supplied IDs, names, descriptions, dates, amounts, and receipt attachment flags from **Appendix B, page 4**. Additional metadata represents explicit information in those descriptions: economy/business class and category-specific day/night counts. No replacement claim set is introduced.


In [2]:
from __future__ import annotations

import asyncio
import copy
import hashlib
import html
import json
import os
import re
import time
from collections import defaultdict
from datetime import date, datetime, timezone
from decimal import Decimal, InvalidOperation
from enum import Enum
from typing import Any, Literal, Protocol

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

POLICIES = {
    "POL-CAT-01": "Eligible categories — Reimbursable when incurred for a documented business purpose: Airfare (economy class only — see POL-AIR-01); Lodging (hotel room charges); Meals (subject to per-diem limits — see POL-PD-01); Ground transport (taxi, rideshare, train, rental car, parking); Conference / registration fees.",
    "POL-CAT-02": "Ineligible items — Never reimbursable; rejected (deducted in full): Alcohol and minibar charges; Spa, gym, and personal entertainment; In-room movies, personal shopping, gifts; Traffic fines, penalties, and late fees; Any personal (non-business) expense.",
    "POL-PD-01": "Meals — Maximum $75 per day. Amounts above the daily cap are deducted; the rest is reimbursed.",
    "POL-PD-02": "Lodging — Maximum $200 per night. Amounts above the nightly cap are deducted; the rest is reimbursed.",
    "POL-PD-03": "Ground transport — Maximum $50 per day. Amounts above the cap are deducted.",
    "POL-AIR-01": "Airfare class — Only economy class airfare is reimbursable. Business/first-class fares are a policy exception and must be routed to Manual Review (not auto-deducted, because a pre-approval may exist).",
    "POL-RCT-01": "Receipt required above $25 — Any single line item greater than $25 requires an attached, itemized receipt. Airfare and lodging always require a receipt regardless of amount.",
    "POL-RCT-02": "Missing receipt handling — If a receipt is missing for an item that requires one, the item is not silently rejected — the claim is routed to Manual Review so the reviewer can request the receipt.",
    "POL-APR-01": "Auto-approve tier — Total ≤ $500: may be auto-approved by the agent if fully compliant.",
    "POL-APR-02": "Manager tier — Total > $500 and ≤ $2,000: eligible for approval, treated as approvable when fully compliant.",
    "POL-APR-03": "Director / Manual-Review tier — Total > $2,000: exceeds the agent's auto-approval authority and must be routed to Manual Review (director approval required), even if otherwise compliant.",
    "POL-TIME-01": "Submission window — Claims must be submitted within 30 days of the expense date. Late claims are routed to Manual Review.",
}
DECISION_GUIDANCE = "Approve: every item eligible, all receipts present, all within per-diem, total within an approvable tier. Partially Approve: claim valid but some amounts exceed caps; reimburse up to the cap. Reject: items ineligible with nothing reimbursable. Manual Review: any ambiguity, policy exception, high value, missing required receipt, or conflicting information. Prefer Manual Review over forcing a decision. Approval thresholds use reimbursable amounts after deductions."
LIMIT_TABLE = {"meals": {"amount": "75.00", "unit": "day", "ref": "POL-PD-01"}, "lodging": {"amount": "200.00", "unit": "night", "ref": "POL-PD-02"}, "ground_transport": {"amount": "50.00", "unit": "day", "ref": "POL-PD-03"}}
SAMPLE_CLAIMS = [
    {"claim_id": "CLM-001", "employee": "A. Rivera", "purpose": "Attend 2-day industry conference (business)", "business_documented": True, "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted": "2026-06-20", "total_claimed": "1110.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "airfare", "description": "Round-trip economy airfare", "amount": "420.00", "receipt_attached": True, "airfare_class": "economy"},
        {"item_id": "I2", "category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": "360.00", "receipt_attached": True, "units": 2},
        {"item_id": "I3", "category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": "180.00", "receipt_attached": True, "units": 3},
        {"item_id": "I4", "category": "conference_fees", "description": "Conference registration", "amount": "150.00", "receipt_attached": True}]},
    {"claim_id": "CLM-002", "employee": "B. Osei", "purpose": "Weekend hotel stay", "business_documented": False, "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted": "2026-06-25", "total_claimed": "380.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "spa", "description": "Hotel spa package", "amount": "300.00", "receipt_attached": True},
        {"item_id": "I2", "category": "minibar", "description": "In-room minibar", "amount": "80.00", "receipt_attached": True}]},
    {"claim_id": "CLM-003", "employee": "C. Nakamura", "purpose": "Client site visit (business)", "business_documented": True, "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted": "2026-06-22", "total_claimed": "940.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "airfare", "description": "Round-trip economy airfare", "amount": "300.00", "receipt_attached": True, "airfare_class": "economy"},
        {"item_id": "I2", "category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": "500.00", "receipt_attached": True, "units": 2},
        {"item_id": "I3", "category": "meals", "description": "Meals, 2 days @ $70/day", "amount": "140.00", "receipt_attached": True, "units": 2}]},
    {"claim_id": "CLM-004", "employee": "D. Fischer", "purpose": "International vendor negotiation (business)", "business_documented": True, "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted": "2026-06-28", "total_claimed": "3000.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "airfare", "description": "Business-class international airfare", "amount": "2400.00", "receipt_attached": True, "airfare_class": "business"},
        {"item_id": "I2", "category": "lodging", "description": "Hotel, 3 nights", "amount": "600.00", "receipt_attached": False, "units": 3}]},
    {"claim_id": "CLM-005", "employee": "E. Haddad", "purpose": "Client dinner / business development", "business_documented": True, "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted": "2026-06-24", "total_claimed": "220.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "meals", "description": "Client dinner for 4 (business development)", "amount": "220.00", "receipt_attached": False, "units": 1}]},
]

## Typed input and exact output contract

Pydantic validates structural input and rejects unknown fields. `Decimal` retains exact cents for calculations. Invalid amounts, duplicate item IDs, and unsupported currencies are rejected. Semantic conflicts such as a total mismatch will be routed to manual review by the policy engine in milestone 2.

`ClaimResult` contains exactly the nine required result fields. The four decision codes are `APPROVE`, `PARTIAL_APPROVE`, `REJECT`, and `MANUAL_REVIEW`. Operational reason codes will appear inside `explanation`, without adding a tenth field.


In [3]:
CENT = Decimal("0.01")


def money(value: Any) -> Decimal:
    if isinstance(value, bool):
        raise ValueError("Boolean is not a monetary amount")
    try:
        result = Decimal(str(value))
    except (InvalidOperation, TypeError):
        raise ValueError("Invalid monetary amount") from None
    if not result.is_finite() or result < 0 or result > 10000000 or result != result.quantize(CENT):
        raise ValueError("USD amount must be finite, non-negative and have at most two decimal places")
    return result.quantize(CENT)


def jsonable(value: Any) -> Any:
    if isinstance(value, BaseModel):
        return jsonable(value.model_dump())
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, dict):
        return {k: jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    return value


def fingerprint(value: Any) -> str:
    return hashlib.sha256(json.dumps(jsonable(value), sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()).hexdigest()


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True)


class Expense(StrictModel):
    item_id: str = Field(min_length=1, max_length=40)
    category: str = Field(min_length=1, max_length=60)
    description: str = Field(min_length=1, max_length=1000)
    amount: Decimal = Field(gt=0)
    receipt_attached: bool
    receipt_itemized: bool = True  # Appendix B's Yes is assumed itemized, not OCR-verified.
    units: int | None = Field(default=None, ge=1, le=366)
    expense_date: date | None = None
    airfare_class: Literal["economy", "business", "first"] | None = None
    _amount = field_validator("amount", mode="before")(money)


class Claim(StrictModel):
    claim_id: str = Field(min_length=1, max_length=60, pattern=r"^[A-Za-z0-9_-]+$")
    employee: str = Field(min_length=1, max_length=100)
    purpose: str = Field(min_length=1, max_length=2000)
    business_documented: bool
    trip_start: date
    trip_end: date
    submitted: date
    currency: Literal["USD"] = "USD"
    total_claimed: Decimal = Field(gt=0)
    items: list[Expense] = Field(min_length=1, max_length=50)
    _total = field_validator("total_claimed", mode="before")(money)

    @model_validator(mode="after")
    def unique_items(self):
        if len({x.item_id for x in self.items}) != len(self.items):
            raise ValueError("Each item_id must be unique within its claim")
        return self


class Decision(str, Enum):
    APPROVE = "APPROVE"
    PARTIAL_APPROVE = "PARTIAL_APPROVE"
    REJECT = "REJECT"
    MANUAL_REVIEW = "MANUAL_REVIEW"


class ClaimResult(StrictModel):
    claim_id: str
    decision: Decision
    approved_amount: Decimal
    deducted_amount: Decimal
    missing_docs: list[str]
    policy_refs: list[str]
    confidence: float = Field(ge=0, le=1)
    explanation: str = Field(min_length=1, max_length=12000)
    tools_used: list[str]
    _amounts = field_validator("approved_amount", "deducted_amount", mode="before")(money)


class Finding(StrictModel):
    code: str
    message: str
    policy_refs: list[str]
    item_ids: list[str] = Field(default_factory=list)
    review: bool = False


def finding(code: str, message: str, refs: list[str], items: list[str] | None = None, review: bool = False) -> dict:
    return Finding(code=code, message=message, policy_refs=refs, item_ids=items or [], review=review).model_dump()


class PolicyRepository:
    """Explicit lookup is sufficient for twelve fixed rules; no vector database."""
    def __init__(self):
        self.rules = dict(POLICIES)
        self.version = fingerprint({"rules": self.rules, "limits": LIMIT_TABLE, "guidance": DECISION_GUIDANCE})

    def lookup(self, refs: list[str] | None = None) -> dict:
        refs = refs or list(self.rules)
        if any(ref not in self.rules for ref in refs):
            raise ValueError("Unknown policy reference")
        return {"policy_version": self.version, "rules": {ref: self.rules[ref] for ref in refs}, "decision_guidance": DECISION_GUIDANCE}

## Provided-input verification

These checks validate transcription and the schema. They do **not** claim that the agent has evaluated reimbursement outcomes yet.


In [4]:
CLAIMS = [Claim.model_validate(raw) for raw in SAMPLE_CLAIMS]
EXPECTED_TOTALS = {"CLM-001": "1110.00", "CLM-002": "380.00", "CLM-003": "940.00", "CLM-004": "3000.00", "CLM-005": "220.00"}
assert len(POLICIES) == 12
assert len(CLAIMS) == 5
assert [c.claim_id for c in CLAIMS] == list(EXPECTED_TOTALS)
for claim in CLAIMS:
    assert claim.total_claimed == Decimal(EXPECTED_TOTALS[claim.claim_id])
    assert sum(item.amount for item in claim.items) == claim.total_claimed
    print(f"{claim.claim_id} · {claim.employee} · ${claim.total_claimed:,.2f} · {len(claim.items)} line items")
assert sum(c.total_claimed for c in CLAIMS) == Decimal("5650.00")
print("Total claimed: $5,650.00. All five supplied input totals reconcile.")


CLM-001 · A. Rivera · $1,110.00 · 4 line items
CLM-002 · B. Osei · $380.00 · 2 line items
CLM-003 · C. Nakamura · $940.00 · 3 line items
CLM-004 · D. Fischer · $3,000.00 · 2 line items
CLM-005 · E. Haddad · $220.00 · 1 line items
Total claimed: $5,650.00. All five supplied input totals reconcile.


## Design Notes & Reasoning

- **Source of truth:** the assignment's policy and five claims; no real employee/company data.
- **Receipt metadata:** a supplied “Yes” is assumed to represent an itemized receipt. No OCR or receipt authenticity verification is implied.
- **Dates and quantities:** retain category-specific day/night counts. Assume expenses occurred during the supplied trip interval when no exact date is supplied. Preserve CLM-004's stated three hotel nights and flag the discrepancy with its trip dates in the next milestone.
- **Manual-review amounts:** hold the entire claim; post neither approval nor final deduction. Show provisional calculations separately, without adding fields to the required result schema.
- **Agent boundary:** the local model will select tools and propose a recommendation. Python will calculate money and validate the proposal. Missing/conflicting evidence must not be turned into an automatic approval.
- **MCP:** the six policy tools will be exposed through the official MCP SDK's in-process client/server transport, preserving a single-file demo.
- **Simplicity:** no database, hosted services, vector index, or companion app is required to run the assessed notebook.
- **Known gaps at this checkpoint:** no agent results, replay recording, dashboard, or completed policy evaluation is included yet. The final notebook will replace this checkpoint status with measured verification evidence.

### References

1. Supplied assignment: *AI Developer Candidate Assignment — Travel Reimbursement Approval Agent*, sections 3–7 and Appendices A–B.
2. [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/).
3. [LangChain ChatOllama integration](https://docs.langchain.com/oss/python/integrations/chat/ollama).
4. [Official MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk).


In [5]:
REQUIRED_FIELDS = {"claim_id", "decision", "approved_amount", "deducted_amount", "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"}
assert set(ClaimResult.model_fields) == REQUIRED_FIELDS
assert {d.value for d in Decision} == {"APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"}
assert jsonable(Decimal("840.00")) == 840.0
print("Milestone 1 verified: 12 policies, 5 exact claims, reconciled totals, and the exact 9-field result schema.")


Milestone 1 verified: 12 policies, 5 exact claims, reconciled totals, and the exact 9-field result schema.
